# SQL-Based Business Analysis (Revenue & Performance)

In this section, we answer key business questions using SQL queries on the 
cleaned dataset stored in MySQL. This simulates a 
real-world analytics workflow where data is queried directly from a database.


In [1]:
import mysql.connector
import pandas as pd

In [2]:
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='',
    database='customer'
)

In [3]:
def run_query(query):
    return pd.read_sql(query, conn)

In [69]:
# Q1. Which customers used a discount but still spent more than the average purchase amount?
run_query(
    '''
        SELECT
            customer_id,
            SUM(purchase_amount) AS total_spent
        FROM customer_shopping_behavior
        WHERE
            discount_applied = 'Yes' 
            GROUP BY customer_id
            HAVING total_spent > (SELECT AVG(purchase_amount) FROM customer_shopping_behavior)
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,customer_id,total_spent
0,2,64.0
1,3,73.0
2,4,90.0
3,7,85.0
4,9,97.0
...,...,...
834,1667,64.0
835,1671,73.0
836,1673,73.0
837,1674,62.0


### Insight

A significant number of customers (**839**) used discounts but still spent **above the average purchase value**.
This indicates that discounts are not only attracting price-sensitive buyers but are also **encouraging higher spending behavior**.

In [38]:
# Q2. Which are the top 5 products with the highest average review rating?
run_query(
    '''
        SELECT
            item_purchased,
            ROUND(AVG(review_rating),2) AS avg_rating
        FROM customer_shopping_behavior
        GROUP BY item_purchased
        ORDER BY avg_rating DESC LIMIT 5;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,item_purchased,avg_rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Handbag,3.78


### Insight

- **Gloves** have the highest customer satisfaction.

- **Footwear items** (**Sandals, Boots**) are performing well in terms of customer experience.

- **Accessories** like **Hat** and **Handbag** also maintain strong ratings.

In [39]:
# Q3. Compare the average Purchase Amounts between Standard and Express Shipping.
run_query(
    '''
        SELECT
            shipping_type,
            ROUND(AVG(purchase_amount),2) AS avg_purchase_value
        FROM customer_shopping_behavior
        WHERE shipping_type IN('Standard', 'Express')
        GROUP BY shipping_type;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,shipping_type,avg_purchase_value
0,Express,60.48
1,Standard,58.46


### Insight

* Customers choosing **Express Shipping** have a slightly higher average purchase value (**60.48**) compared to **Standard Shipping** (**58.46**).
* This indicates that customers opting for faster delivery tend to spend **more per order**.
* **Express Shipping** customers may represent **high-value or urgent buyers**, making them a good target for **premium offers or priority services**.

In [40]:
# Q4. Do subscribed customers spend more? Compare average spend and total revenue between subscribers and non-subscribers.
run_query(
    '''
        SELECT
            subscription_status,
            SUM(purchase_amount) AS total_revenue,
            ROUND(AVG(purchase_amount),2) AS avg_spend
        FROM customer_shopping_behavior
        GROUP BY subscription_status;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,subscription_status,total_revenue,avg_spend
0,No,170436.0,59.87
1,Yes,62645.0,59.49


### Insight

* **Non-subscribed customers** contribute significantly more to total revenue (**170,436**) compared to **subscribed customers** (**62,645**).
* The **average spend per customer** is also slightly higher for **non-subscribers (59.87)** than for **subscribers (59.49)**.
* This indicates that the subscription program is **not currently driving higher spending**, and its value proposition may need improvement to increase customer engagement and purchase behavior.


In [41]:
# Q5. Which 5 products have the highest percentage of purchases with discounts applied?
run_query(
    '''
        SELECT
            item_purchased,
            discount_applied,
            ROUND(SUM(purchase_amount) / 
            (SELECT SUM(purchase_amount) 
            FROM customer_shopping_behavior
            WHERE discount_applied = 'Yes') * 100, 2) AS highest_percentage_of_purchase
        FROM customer_shopping_behavior
        WHERE discount_applied = 'Yes'
        GROUP BY item_purchased
        ORDER BY highest_percentage_of_purchase DESC LIMIT 5;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,item_purchased,discount_applied,highest_percentage_of_purchase
0,Pants,Yes,4.87
1,Sweater,Yes,4.64
2,Hat,Yes,4.62
3,Coat,Yes,4.50
4,Dress,Yes,4.39


### Insight

* **Pants (4.87%)**, **Sweater (4.64%)**, **Hat (4.62%)**, **Coat (4.50%)**, and **Dress (4.39%)** have the highest proportion of purchases made with discounts.
* This suggests that these products are **highly price-sensitive**, and customers are more likely to purchase them when promotions are offered.
* Targeted discount campaigns on these categories can be an **effective strategy to boost sales volume and clear inventory**.


In [42]:
# Q6. Segment customers into New, Returning, and Loyal based on their total number of previous purchases, and show the count of each segment.
run_query(
    '''
        WITH customer_type AS(
            SELECT
                customer_id,
                CASE
                    WHEN previous_purchases = 1 THEN 'New'
                    WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
                    ELSE 'Loyal'
                END AS customer_segment
            FROM customer_shopping_behavior
        )

        SELECT
            customer_segment,
            COUNT(customer_id) AS total_customer
        FROM customer_type
        GROUP BY customer_segment;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,customer_segment,total_customer
0,Loyal,3116
1,New,83
2,Returning,701


### Insight

* The customer base is dominated by **Loyal customers (3,116)**, indicating a **strong repeat purchase behavior** and high customer retention.
* **Returning customers (701)** represent a moderate opportunity for **loyalty-building through targeted engagement and personalized offers**.
* The relatively small number of **New customers (83)** suggests a need to **strengthen acquisition strategies** to expand the customer base and support long-term growth.


In [43]:
# Q7. What are the top 3 most purchased products within each category?
run_query(
    '''
        WITH top_products AS(
            SELECT
                item_purchased,
                category,
                COUNT(customer_id) AS total_orders,
                ROW_NUMBER() OVER(partition by category ORDER BY COUNT(customer_id) DESC) AS ranked_item
            FROM customer_shopping_behavior
            GROUP BY item_purchased, category
        )
        SELECT
            item_purchased,
            category,
            total_orders,
            ranked_item
        FROM top_products
        WHERE ranked_item <= 3
        ORDER BY category, ranked_item
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,item_purchased,category,total_orders,ranked_item
0,Jewelry,Accessories,171,1
1,Sunglasses,Accessories,161,2
2,Belt,Accessories,161,3
3,Blouse,Clothing,171,1
4,Pants,Clothing,171,2
5,Shirt,Clothing,169,3
6,Sandals,Footwear,160,1
7,Shoes,Footwear,150,2
8,Sneakers,Footwear,145,3
9,Jacket,Outerwear,163,1


### Insight

* In **Accessories**, **Jewelry, Sunglasses, and Belt** are the most purchased items, indicating strong demand for **fashion add-ons and style-enhancing products**.
* For **Clothing**, **Blouse, Pants, and Shirt** lead the sales, highlighting **everyday wear essentials** as key revenue drivers.
* In **Footwear**, **Sandals, Shoes, and Sneakers** dominate purchases, reflecting a preference for **comfortable and versatile footwear**.
* Within **Outerwear**, **Jacket and Coat** are the top-performing items, suggesting consistent demand for **seasonal and protective wear**.
* These top products represent **core demand drivers**, making them ideal candidates for **inventory prioritization, promotions, and cross-selling strategies**.

In [44]:
# Q8. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?
run_query(
    '''
        SELECT
            COUNT(customer_id) AS repeat_buyers,
            subscription_status
        FROM customer_shopping_behavior
        WHERE previous_purchases > 5
        GROUP BY subscription_status
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,repeat_buyers,subscription_status
0,2518,No
1,958,Yes


### Insight

* Among repeat buyers (more than 5 previous purchases), a significantly larger portion (**2,518 customers**) are **non-subscribers**, while only **958 customers** have subscribed.
* This indicates that **highly engaged and loyal customers are not automatically converting into subscribers**.
* The business has a strong opportunity to **target repeat buyers with subscription-specific incentives**, such as exclusive discounts, early access, or loyalty benefits, to **increase subscription adoption and long-term customer value**.


In [45]:
# Q9: What is the total revenue and average purchase value?
run_query(
    '''
        SELECT 
            SUM(purchase_amount) as total_revenue,
            AVG(purchase_amount) as avg_purchase_value
        FROM customer_shopping_behavior
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,total_revenue,avg_purchase_value
0,233081.0,59.7644


### Insight

The business has generated a **total revenue** of **233,081**, with an **average purchase 
value** of approximately **59.76** per transaction. This indicates stable and 
consistent customer spending behavior, providing a strong baseline for further 
segment-level revenue analysis.


In [46]:
# Q10: Which product categories contribute the most to overall revenue?
run_query(
    '''
        SELECT 
            category, 
            SUM(purchase_amount) AS total_revenue 
        FROM customer_shopping_behavior 
        GROUP BY category
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,category,total_revenue
0,Clothing,104264.0
1,Accessories,74200.0
2,Footwear,36093.0
3,Outerwear,18524.0


### Insight

**Clothing** is the highest revenue-generating category, contributing nearly half of 
the total revenue. **Accessories** follow as the second-largest contributor, while 
***Footwear*** and ***Outerwear*** generate comparatively lower sales.


In [47]:
# Q11: List Top 10 locations generates the highest total revenue?
run_query(
    '''
        SELECT 
            location, 
            SUM(purchase_amount) AS total_revenue 
        FROM customer_shopping_behavior 
        GROUP BY location
        ORDER BY total_revenue DESC limit 10;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,location,total_revenue
0,Montana,5784.0
1,Illinois,5617.0
2,California,5605.0
3,Idaho,5587.0
4,Nevada,5514.0
5,Alabama,5261.0
6,New York,5257.0
7,North Dakota,5220.0
8,West Virginia,5174.0
9,Nebraska,5172.0


### Insight

Revenue is distributed across multiple locations, with **Montana**, **Illinois**, and 
**California** emerging as the top revenue-generating regions. The relatively close 
revenue values among top locations suggest a well-diversified geographic market 
rather than dependency on a single region.


In [48]:
# Q12: What percentage of total revenue does each category contribute?
run_query(
    '''
        SELECT 
            category,
            SUM(purchase_amount) AS category_revenue,
            ROUND((SUM(purchase_amount) * 100 / SUM(SUM(purchase_amount)) OVER()), 2) AS percentage_contribution
        FROM customer_shopping_behavior
        GROUP BY category
        ORDER BY percentage_contribution DESC;
        
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,category,category_revenue,percentage_contribution
0,Clothing,104264.0,44.73
1,Accessories,74200.0,31.83
2,Footwear,36093.0,15.49
3,Outerwear,18524.0,7.95


### Insight

**Clothing** dominates overall revenue, contributing approximately **44.73%** of total 
sales, followed by **Accessories** at **31.83%**. ***Footwear*** and ***Outerwear*** together account 
for less than **25%** of revenue, indicating potential growth opportunities in these 
categories.


In [49]:
# Q13: Which gender type contributes the most to total revenue?
run_query(
    '''
        SELECT
            gender,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY gender
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,gender,total_revenue
0,Male,157890.0
1,Female,75191.0


### Insight

**Male** customers contribute significantly **higher total revenue** than **female** customers, generating more than double the revenue.
This is primarily driven by a larger customer base and higher purchase volume, making males the dominant revenue-driving segment overall.

In [50]:
# Q14: What is the average purchase value of gender type?
run_query(
    '''
        SELECT
            gender,
            AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY gender
        ORDER BY avg_purchase_value DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,gender,avg_purchase_value
0,Female,60.2492
1,Male,59.5362


### Insight

**Female** customers have a slightly **higher average purchase value** compared to **male** customers.
This indicates that while females contribute less in total revenue, they tend to place higher-value orders per transaction.

In [51]:
# Q15: Which gender type contributes the most in each category?
run_query(
    '''
        SELECT
            gender,
            category,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY gender, category
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,gender,category,total_revenue
0,Male,Clothing,70628.0
1,Male,Accessories,50381.0
2,Female,Clothing,33636.0
3,Male,Footwear,24258.0
4,Female,Accessories,23819.0
5,Male,Outerwear,12623.0
6,Female,Footwear,11835.0
7,Female,Outerwear,5901.0


### Insight

**Male** customers dominate total revenue across all product categories, with the strongest contribution coming from **Clothing and Accessories**.
**Female** customers also show meaningful contributions, particularly in **Clothing and Accessories**, but at lower overall revenue levels.

In [52]:
# Q16: Which age group contributes the most to total revenue?
run_query(
    '''
        SELECT
            age_group,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY age_group
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,age_group,total_revenue
0,Middle Aged,89445.0
1,Adult,65842.0
2,Senior,43164.0
3,Young Adult,34630.0


### Insight

**Middle-aged** customers generate the highest revenue, followed by **adults**. ***Seniors*** 
and ***young adults*** contribute comparatively lower revenue, indicating that the core 
revenue-driving segment lies within the middle-aged demographic.


In [53]:
# Q17: Which age group and category combinations generate the highest revenue?
run_query(
    '''
        SELECT
            age_group,
            category,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY age_group, category
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,age_group,category,total_revenue
0,Middle Aged,Clothing,39758.0
1,Adult,Clothing,28864.0
2,Middle Aged,Accessories,27179.0
3,Adult,Accessories,22000.0
4,Senior,Clothing,19421.0
5,Young Adult,Clothing,16221.0
6,Middle Aged,Footwear,15097.0
7,Senior,Accessories,14510.0
8,Young Adult,Accessories,10511.0
9,Adult,Footwear,10195.0


### Insight

**Middle-aged** customers purchasing **Clothing** generate the highest revenue, followed 
by **adults** in the same category. **Accessories** also perform strongly among **middle-aged** 
and **adult** customers, highlighting these segments as the most valuable for targeted 
marketing and inventory planning.


In [54]:
# Q18: Which age group contributes the most revenue in each location?
run_query(
    '''
        SELECT
            location,
            age_group,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY location, age_group
        ORDER BY location, total_revenue DESC LIMIT 10;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,location,age_group,total_revenue
0,Alabama,Middle Aged,2230.0
1,Alabama,Adult,1153.0
2,Alabama,Young Adult,989.0
3,Alabama,Senior,889.0
4,Alaska,Middle Aged,1754.0
5,Alaska,Adult,1693.0
6,Alaska,Senior,845.0
7,Alaska,Young Adult,575.0
8,Arizona,Middle Aged,1965.0
9,Arizona,Adult,1314.0


### Insight

Across most locations, **middle-aged** customers contribute the highest revenue, 
followed by **adult** customers. This pattern indicates that middle-aged consumers are 
the dominant revenue-driving segment across geographies.


In [55]:
# Q19: Which combined customer segments (Location × Age Group × Category) generate the highest revenue?
run_query(
    '''
        SELECT
            location,
            age_group,
            category,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY location, age_group, category
        ORDER BY total_revenue DESC LIMIT 10;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,location,age_group,category,total_revenue
0,Alabama,Middle Aged,Clothing,1286.0
1,Montana,Middle Aged,Clothing,1157.0
2,Ohio,Middle Aged,Clothing,1144.0
3,Wyoming,Middle Aged,Clothing,1114.0
4,Arizona,Middle Aged,Clothing,1092.0
5,Utah,Middle Aged,Clothing,1083.0
6,Rhode Island,Middle Aged,Clothing,1066.0
7,Nebraska,Middle Aged,Clothing,1042.0
8,Idaho,Middle Aged,Clothing,1039.0
9,Vermont,Middle Aged,Clothing,1023.0


### Insight

**Middle-aged** customers purchasing **Clothing** consistently generate the highest 
revenue across multiple locations. This segment represents the most valuable 
customer group and should be prioritized for targeted promotions and retention 
strategies.


In [56]:
# Q20: Which purchase frequency segment generates the highest revenue?
run_query(
    '''
        SELECT
            frequency_of_purchases,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY frequency_of_purchases
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,frequency_of_purchases,total_revenue
0,Every 3 Months,35088.0
1,Annually,34419.0
2,Quarterly,33771.0
3,Bi-Weekly,33200.0
4,Monthly,32810.0
5,Fortnightly,32007.0
6,Weekly,31786.0


### Insight

Customers purchasing **every three months** and **annually** contribute the highest 
revenue. This indicates that moderate-frequency customers generate more total 
revenue than very frequent shoppers, likely due to higher purchase values.


In [57]:
# Q21: Do frequent customers spend more or less per transaction?
run_query(
    '''
        SELECT
            frequency_of_purchases,
            AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY frequency_of_purchases
        ORDER BY avg_purchase_value DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,frequency_of_purchases,avg_purchase_value
0,Bi-Weekly,60.6947
1,Annually,60.1731
2,Every 3 Months,60.0822
3,Quarterly,59.9840
4,Monthly,59.3309
5,Fortnightly,59.0535
6,Weekly,58.9722


### Insight

**Bi-weekly** and **annually** purchasing customers have the highest average purchase 
values, while **weekly** customers spend slightly less per transaction. This suggests 
that higher purchase frequency does not always translate to higher spend per order.


In [58]:
# Q22: Which customer segments combine high purchase frequency with high revenue?
run_query(
    '''
        SELECT
            frequency_of_purchases,
            SUM(purchase_amount) AS total_revenue,
            AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY frequency_of_purchases
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,frequency_of_purchases,total_revenue,avg_purchase_value
0,Every 3 Months,35088.0,60.0822
1,Annually,34419.0,60.1731
2,Quarterly,33771.0,59.9840
3,Bi-Weekly,33200.0,60.6947
4,Monthly,32810.0,59.3309
5,Fortnightly,32007.0,59.0535
6,Weekly,31786.0,58.9722


### Insight

Customers purchasing **every three months** and **annually** generate the **highest total 
revenue**, while **bi-weekly** customers show the **highest average purchase value**. This 
indicates multiple high-value loyalty segments with different purchasing behaviors.


In [59]:
# Q23: Does offering discounts increase overall revenue?
run_query(
    '''
        SELECT
            discount_applied,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY discount_applied
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,discount_applied,total_revenue
0,No,133670.0
1,Yes,99411.0


### Insight

Purchases **without discounts** generate **higher total revenue** compared to **discounted 
purchases**. This suggests that **discounts** are not the primary driver of overall 
revenue and should be applied strategically rather than broadly.


In [60]:
# Q24: Do discounts increase the average purchase value?
run_query(
    '''
        SELECT
            discount_applied,
            AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY discount_applied
        ORDER BY avg_purchase_value DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,discount_applied,avg_purchase_value
0,No,60.1305
1,Yes,59.2791


### Insight

The **average purchase value** is slightly **higher** for **non-discounted purchases**, 
indicating that discounts do not significantly increase spending per transaction.


In [61]:
# Q25: In which product categories are discounts most effective?
run_query(
    '''
        SELECT
            category,
            discount_applied,
            AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY category, discount_applied
        ORDER BY avg_purchase_value DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,category,discount_applied,avg_purchase_value
0,Footwear,Yes,61.7992
1,Accessories,No,60.8895
2,Clothing,No,60.2237
3,Clothing,Yes,59.7524
4,Footwear,No,59.0794
5,Outerwear,No,58.6556
6,Accessories,Yes,58.4899
7,Outerwear,Yes,55.3194


### Insight

**Discounts** appear to be most effective for **Footwear**, where discounted purchases 
have a higher average value than non-discounted ones. In contrast, discounts reduce 
average purchase value for Clothing, Accessories, and Outerwear, suggesting that 
discount strategies should be category-specific.


In [62]:
# Q26: Which product categories receive the highest customer ratings?
run_query(
    '''
        SELECT
            category,
            ROUND(AVG(review_rating),2) AS avg_review_rating
        FROM customer_shopping_behavior
        GROUP BY category
        ORDER BY review_rating DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,category,avg_review_rating
0,Accessories,3.77
1,Footwear,3.79
2,Clothing,3.72
3,Outerwear,3.75


### Insight

Average customer ratings are fairly consistent across all product categories, 
ranging between **3.7 and 3.8** **Footwear** receives the **highest average rating**, while 
**Clothing** has the **lowest**, though differences are marginal, indicating broadly 
similar customer satisfaction levels across categories.


In [63]:
# Q27: Do higher-rated products generate more revenue?
run_query(
    '''
        SELECT
            CASE
                WHEN review_rating < 3 THEN 'Low Rating'
                WHEN review_rating BETWEEN 3 AND 4 THEN 'MEDIUM Rating'
                ELSE 'HIGH RATING'
            END AS rating_band,
        SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY rating_band
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,rating_band,total_revenue
0,MEDIUM Rating,105122.0
1,HIGH RATING,88173.0
2,Low Rating,39786.0


### Insight

Purchases with **medium ratings** generate the **highest total revenue**, followed by 
high-rated purchases. **Low-rated** purchases contribute the **least to revenue**, 
indicating that while high satisfaction is valuable, the majority of revenue comes 
from moderately satisfied customers.


In [64]:
# Q28: Do higher customer ratings lead to higher spending per transaction?
run_query(
    '''
        SELECT
            CASE
                WHEN review_rating < 3 THEN 'Low Rating'
                WHEN review_rating BETWEEN 3 AND 4 THEN 'MEDIUM Rating'
                ELSE 'HIGH RATING'
            END AS rating_band,
        AVG(purchase_amount) AS avg_purchase_value
        FROM customer_shopping_behavior
        GROUP BY rating_band
        ORDER BY avg_purchase_value DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,rating_band,avg_purchase_value
0,HIGH RATING,60.6834
1,MEDIUM Rating,59.4918
2,Low Rating,58.5088


### Insight

**Higher-rated** purchases have the **highest average purchase value**, indicating that 
customer satisfaction positively influences spending per transaction, even though 
most revenue comes from medium-rated purchases.


In [65]:
# Q29: Which seasons generate the highest revenue?
run_query(
    '''
        SELECT
            season,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY season
        ORDER BY total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,season,total_revenue
0,Fall,60018.0
1,Spring,58679.0
2,Winter,58607.0
3,Summer,55777.0


### Insight

Revenue remains relatively stable across all seasons, with **Fall** generating the 
highest sales. This indicates consistent year-round demand with a slight seasonal 
uplift during Fall.


In [66]:
# Q30: Which product categories perform best in each season?
run_query(
    '''
        SELECT
            season,
            category,
            SUM(purchase_amount) AS total_revenue
        FROM customer_shopping_behavior
        GROUP BY season, category
        ORDER BY season, total_revenue DESC;
    '''
)

C:\Users\mohda\AppData\Local\Temp\ipykernel_11604\254819045.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,season,category,total_revenue
0,Fall,Clothing,26220.0
1,Fall,Accessories,19874.0
2,Fall,Footwear,8665.0
3,Fall,Outerwear,5259.0
4,Spring,Clothing,27692.0
5,Spring,Accessories,17007.0
6,Spring,Footwear,9555.0
7,Spring,Outerwear,4425.0
8,Summer,Clothing,23078.0
9,Summer,Accessories,19028.0


### Insight

**Clothing** consistently generates the **highest revenue** across all **seasons**, followed 
by **Accessories**. **Footwear** and **Outerwear** show lower but stable performance, with no 
strong seasonal dependency, indicating steady demand throughout the year.


## Final Insights Summary

- Clothing is the primary revenue driver across categories, seasons, and customer segments.
- Middle-aged customers form the most valuable demographic group.
- Moderate-frequency customers generate higher total revenue than very frequent buyers.
- Discounts are effective mainly for Footwear but reduce order value in other categories.
- Customer satisfaction positively impacts average purchase value.
- Revenue remains stable across seasons with a slight peak during Fall.


## Business Recommendations

- Focus marketing and inventory efforts on Clothing and Accessories.
- Design targeted campaigns for middle-aged customers.
- Apply discounts selectively, especially for Footwear.
- Improve product quality or positioning for lower-rated categories like Outerwear.
- Maintain year-round sales strategies due to stable seasonal demand.
